# 0500 · Seudonimización HMAC-SHA256 antes de compartir

**Para qué.** Iván pidió (P9, 25-10) que lo que salga del equipo use HMAC-SHA256 con clave secreta en vez de `md5(valor || sal)`. Este cuaderno toma las tablas por guía que se vayan a entregar (jurado, asesor, anexos) y reemplaza los identificadores por su HMAC-SHA256. **No cambia ningún resultado** ni obliga a volver a correr el pipeline: el seudónimo se calcula sobre el valor que ya hay en la tabla (el md5 de 0300/0330).

| Identificador | Hoy en el Volume | En la copia para compartir |
| --- | --- | --- |
| `guia` | `md5(guía \|\| sal)`, sal en `SEGURIDAD/sal_guia.txt` (0330) | `HMAC-SHA256(clave, md5)` |
| `ID_CLIENTE` | `md5(ID_CLIENTE \|\| SAL_CLIENTE)` (0300, pos. 9); **la sal quedó escrita en el cuaderno** | `HMAC-SHA256(clave, md5)` |

**La clave.** 32 bytes aleatorios. Se lee del secreto de Databricks (`SCOPE_SECRETO`/`NOMBRE_SECRETO`) si existe; si no, de `SEGURIDAD/clave_hmac.txt` en el Volume, que el cuaderno crea la primera vez. La clave **nunca** se imprime, no se copia al repositorio y no se entrega con las tablas. Con la misma clave, el mismo valor da el mismo seudónimo en todas las tablas, así que los cruces entre tablas compartidas siguen funcionando.

**Escribe** solo en `V3/COMPARTIR/`: una copia de cada tabla y `P11_manifiesto_compartir.csv` (filas, columnas, controles). Las tablas originales no se tocan.

**Controles (el cuaderno se detiene si alguno falla).**
1. Cada columna seudonimizada conserva el mismo número de valores distintos (no hay colisiones).
2. Ningún valor original aparece en la copia.
3. La copia no trae columnas de la lista `PROHIBIDAS` (número de guía real, nombres, teléfonos, direcciones, correos).

In [ ]:
%pip install "duckdb<2"

In [ ]:
import os, json, datetime
import pandas as pd
from MODULOS.seudonimo import leer_o_crear_clave, seudonimizar, ALGORITMO

PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_V3      = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_SALIDA  = f"{CARPETA_V3}COMPARTIR/"
RUTA_CLAVE      = f"{PATH_DATABRICKS}SEGURIDAD/clave_hmac.txt"     # fuera de Git
SCOPE_SECRETO, NOMBRE_SECRETO = "proyectogrado", "clave_hmac"      # opcional: secreto de Databricks

# Tablas por guía que se van a compartir (ruta relativa a V3/). Agregar o quitar según lo que se entregue.
TABLAS = [
    "MODELADO/base_modelado.parquet",            # 0411: base de modelado (guías con margen en T0)
    "P4b_predicciones_validacion.parquet",       # 0416: predicciones de validación (abril a junio)
]
COLUMNAS_ID = ["guia", "ID_CLIENTE"]
PROHIBIDAS  = {"NOMBRE", "NOMBRES", "TELEFONO", "DIRECCION", "CORREO", "EMAIL", "CELULAR"}   # palabras dentro del nombre de la columna
PROHIBIDAS_EXACTAS = {"NUMERO_GUIA", "GUIA_REAL"}

In [ ]:
# 1. Clave (no se imprime)
try:
    CLAVE = leer_o_crear_clave(RUTA_CLAVE, dbutils=dbutils, scope=SCOPE_SECRETO, nombre=NOMBRE_SECRETO)
    origen_clave = "secreto de Databricks"
except Exception:
    CLAVE = leer_o_crear_clave(RUTA_CLAVE)
    origen_clave = "archivo del Volume (SEGURIDAD/)"
print(f"Clave de {len(CLAVE)} bytes leída desde {origen_clave}")

In [ ]:
# 2. Seudonimizar y guardar cada tabla
os.makedirs(CARPETA_SALIDA, exist_ok=True)
manifiesto = []
for rel in TABLAS:
    ruta = f"{CARPETA_V3}{rel}"
    if not os.path.exists(ruta):
        print(f"No existe, se omite: {rel}")
        continue
    t = pd.read_parquet(ruta)
    prohibidas = [c for c in t.columns if c.upper() in PROHIBIDAS_EXACTAS or set(c.upper().split("_")) & PROHIBIDAS]
    assert not prohibidas, f"{rel} trae columnas que no se pueden compartir: {prohibidas}"
    cols = [c for c in COLUMNAS_ID if c in t.columns]
    t2, ctl = seudonimizar(t, cols, CLAVE)
    for _, r in ctl[ctl.presente].iterrows():
        assert r.distintos_antes == r.distintos_despues, f"{rel}.{r.columna}: colisión de seudónimos"
        assert r.valores_originales_que_quedan == 0, f"{rel}.{r.columna}: quedan valores originales"
    destino = f"{CARPETA_SALIDA}{os.path.basename(rel)}"
    t2.to_parquet(destino, index=False)
    manifiesto.append({"tabla": rel, "destino": destino.replace(PATH_DATABRICKS, ""), "filas": len(t2),
                       "columnas": len(t2.columns), "seudonimizadas": ",".join(cols), "algoritmo": ALGORITMO,
                       "clave_desde": origen_clave, "fecha": datetime.datetime.now().isoformat(timespec="seconds")})
    print(f"{rel}: {len(t2):,} filas · seudonimizadas {cols}")
    display(ctl)
    del t, t2

In [ ]:
# 3. Manifiesto
man = pd.DataFrame(manifiesto)
man.to_csv(f"{CARPETA_SALIDA}P11_manifiesto_compartir.csv", index=False)
display(man)

## Qué entregar

- Solo lo que está en `V3/COMPARTIR/` y el manifiesto. Nunca `SEGURIDAD/` ni las tablas originales del Volume.
- Si el jurado necesita cruzar dos tablas, ambas deben salir de este cuaderno con la misma clave.
- Para una corrida nueva del pipeline desde cero, lo recomendable es aplicar HMAC-SHA256 desde 0300 y 0330 (ver `docs/seudonimizacion.md`); para los resultados del documento se mantiene el md5 con el que se corrieron.